In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
import sklearn
import seaborn as sns

In [ ]:
from datasets import load_dataset

REPO = "CMU-OSPO/protostar-climate-train-to-2018"
daily = load_dataset(REPO, "daily", split="train").to_pandas()
annual = load_dataset(REPO, "annual_indices", split="train").to_pandas()
cities = load_dataset(REPO, "city_info", split="train").to_pandas()

README.md:   0%|          | 0.00/7.53k [00:00<?, ?B/s]

daily/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 49.3MB            

daily/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10245754 [00:00<?, ? examples/s]

annual_indices/train-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B /  739kB            

annual_indices/train-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/28052 [00:00<?, ? examples/s]

city_info/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 22.1kB            

city_info/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/461 [00:00<?, ? examples/s]

In [ ]:
daily.head()

,ID,Name,Date,tmax,tmin,prcp
0,USW00014929,Aberdeen,1893-01-01,30.0,8.0,0.0
1,USW00014929,Aberdeen,1893-01-02,NaN,-10.0,0.0
2,USW00014929,Aberdeen,1893-01-03,24.0,-3.0,NaN
3,USW00014929,Aberdeen,1893-01-04,37.0,18.0,0.0
4,USW00014929,Aberdeen,1893-01-05,28.0,-10.0,0.0


In [ ]:
annual.head()

,Name,ID,Lat,Lon,Year,Miss Tmax,Miss Tmin,Miss Prcp,Avg Temp,Avg Tmax,...,Max Tmin,Min Tmin,Warm Days,Warm Nights,Cold Days,Cold Nights,Max 1-day P,Max 5-day P,Wet Days,cdd
0,Aberdeen,USW00014929,45.4558,-98.4131,1893,18,17,34,41.09,55.14,...,72.0,-42.0,9.32,5.75,13.70,16.71,1.9,3.31,15.39,591.0
1,Aberdeen,USW00014929,45.4558,-98.4131,1894,127,127,64,49.05,62.58,...,75.0,-19.0,10.68,5.48,1.64,3.56,2.0,3.22,14.74,588.0
2,Aberdeen,USW00014929,45.4558,-98.4131,1895,11,9,0,41.94,56.58,...,70.0,-46.0,7.12,4.11,13.15,16.99,2.5,4.20,21.77,499.5
3,Aberdeen,USW00014929,45.4558,-98.4131,1896,7,7,0,41.16,53.84,...,68.0,-28.0,10.41,4.93,14.79,16.16,2.0,3.77,35.76,606.5
4,Aberdeen,USW00014929,45.4558,-98.4131,1897,11,15,2,40.89,53.71,...,76.0,-34.0,8.77,4.93,18.08,19.73,3.0,4.65,30.39,566.5


In [ ]:
cities.head()

,Name,ID,Lat,Lon,Stn.Name,Stn.stDate,Stn.edDate
0,Lander,USW00024021,42.8153,-108.7261,LANDER WBO,1892-01-01,1946-05-28
1,Lander,USW00024021,42.8153,-108.7261,LANDER HUNT FIELD,1946-05-29,2018-12-31
2,Cheyenne,USW00024018,41.1519,-104.8061,CHEYENNE WBO,1871-01-01,1935-08-31
3,Cheyenne,USW00024018,41.1519,-104.8061,CHEYENNE MUNICIPAL ARPT,1935-09-01,2018-12-31
4,Wausau,USW00014897,44.9258,-89.6256,Wausau Record Herald,1896-01-01,1941-12-31


In [ ]:
TARGETS = {
    "Avg Temp": ["Miss Tmax", "Miss Tmin"],
    "ToT Prcp": ["Miss Prcp"],
    "Max Tmax": ["Miss Tmax"],
    "cdd":      ["Miss Tmax", "Miss Tmin"],
}

df = annual.copy()
for t, miss in TARGETS.items():
    df.loc[(df[miss] > 10).any(axis=1), t] = np.nan

df = df[df["Year"] >= 1950].reset_index(drop=True)
print(df[list(TARGETS)].isna().mean())

def time_split(d, cutoff, horizon=7):
    train = d[d["Year"] <= cutoff]
    val   = d[(d["Year"] > cutoff) & (d["Year"] <= cutoff + horizon)]
    return train, val

folds = {"A": time_split(df, 2011), "B": time_split(df, 2004)}
for k, (tr, va) in folds.items():
    print(k, tr["Year"].agg(["min", "max"]).tolist(), va["Year"].agg(["min", "max"]).tolist(), len(tr), len(va))

Avg Temp    0.057557
ToT Prcp    0.051415
Max Tmax    0.052312
cdd         0.057557
dtype: float64
A [1950, 2011] [2012, 2018] 13020 1470
B [1950, 2004] [2005, 2011] 11550 1470
